# 🧠 DINO World Model — From Scratch

**Paper:** [DINO-WM: World Models on Pre-trained Visual Features enable Zero-shot Planning](https://arxiv.org/abs/2411.04983)  
**Authors:** Gaoyue Zhou, Hengkai Pan, Yann LeCun, Lerrel Pinto (NYU & Meta AI)

---

## What's the big idea?

Most world models learn **everything** from scratch — they learn to see, learn physics, and learn to plan, all at once. This is incredibly data-hungry.

**DINO-WM flips this:** it uses a **frozen** pre-trained vision model (DINOv2) as its eyes, and only learns the **dynamics** ("if I take this action, what happens next?"). Because DINOv2 already understands visual structure, the world model needs far less data and can **plan to reach goals zero-shot**.

### Architecture at a glance:

```
Image → [Frozen DINO Encoder] → patch embeddings
                                      ↓
          action + proprio → [Small Encoders] → action/proprio embeddings
                                      ↓
       [concat all embeddings] → [Transformer Predictor] → predicted next embeddings
                                                                    ↓
                                                          [VQ-VAE Decoder] → predicted next image
```

### What we'll build in this notebook:

1. **A simple environment** — a colored ball moving in a 2D arena
2. **Frozen DINO encoder** — extract rich visual features from raw pixels
3. **Action & proprioceptive encoders** — embed low-dimensional control signals
4. **Transformer predictor** — predict next-step embeddings given history
5. **VQ-VAE decoder** — decode embeddings back to images
6. **Training loop** — train the dynamics model end-to-end
7. **Rollout & visualization** — predict future frames and see results
8. **CEM planner** — plan actions to reach a goal image (zero-shot!)

**Runtime:** ~15 minutes on a T4 GPU (Colab free tier)

---
## 0. Setup & Imports

Make sure you're on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from einops import rearrange
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 1. The Environment: Moving Ball Arena

Before building a world model, we need a world! We'll create a simple 2D environment:
- A **colored ball** moves in a bounded arena
- **Actions** are 2D velocity commands `[dx, dy]`
- **Observations** are 224×224 RGB images (the size DINO expects)
- **Proprioception** is the ball's `[x, y]` position

This is analogous to the **PointMaze** environment used in the paper — simple enough to train quickly, but complex enough to need a real world model.

In [ ]:
class BallArena:
    """Simple 2D environment: a colored ball moves in a bounded arena."""

    def __init__(self, img_size=224, ball_radius=12):
        self.img_size = img_size
        self.ball_radius = ball_radius
        self.pos = np.array([img_size / 2, img_size / 2], dtype=np.float32)
        # Warm orange ball on dark background — easy to see
        self.ball_color = np.array([0.85, 0.47, 0.34])  # #D97757 accent
        self.bg_color = np.array([0.10, 0.10, 0.08])    # dark background

    def reset(self, pos=None):
        """Reset ball to a random or specified position."""
        if pos is not None:
            self.pos = np.array(pos, dtype=np.float32)
        else:
            margin = self.ball_radius + 5
            self.pos = np.random.uniform(
                margin, self.img_size - margin, size=2
            ).astype(np.float32)
        return self._get_obs()

    def step(self, action):
        """Apply action [dx, dy] and return new observation."""
        action = np.array(action, dtype=np.float32)
        self.pos = self.pos + action * 8.0  # scale for visible movement
        # Clamp to arena bounds
        margin = self.ball_radius
        self.pos = np.clip(self.pos, margin, self.img_size - margin)
        return self._get_obs()

    def _get_obs(self):
        """Render the current state as a 224x224 RGB image."""
        img = np.full((self.img_size, self.img_size, 3), self.bg_color, dtype=np.float32)
        # Draw ball using distance field (smooth circle)
        yy, xx = np.mgrid[:self.img_size, :self.img_size]
        dist = np.sqrt((xx - self.pos[0])**2 + (yy - self.pos[1])**2)
        mask = np.clip(1.0 - (dist - self.ball_radius + 1.5) / 1.5, 0, 1)
        for c in range(3):
            img[:, :, c] = img[:, :, c] * (1 - mask) + self.ball_color[c] * mask
        return {
            'visual': img,          # (H, W, 3) float32 in [0, 1]
            'proprio': self.pos.copy()  # (2,) float32
        }


def collect_trajectories(n_trajs=200, traj_len=20, seed=42):
    """Collect random trajectories from the BallArena."""
    np.random.seed(seed)
    env = BallArena()
    all_visuals, all_proprios, all_actions = [], [], []

    for _ in tqdm(range(n_trajs), desc="Collecting trajectories"):
        obs = env.reset()
        visuals = [obs['visual']]
        proprios = [obs['proprio']]
        actions = []

        for t in range(traj_len - 1):
            # Random walk with momentum for smoother trajectories
            action = np.random.randn(2).astype(np.float32) * 1.5
            obs = env.step(action)
            visuals.append(obs['visual'])
            proprios.append(obs['proprio'])
            actions.append(action)

        # Pad last action (not used in prediction)
        actions.append(np.zeros(2, dtype=np.float32))

        all_visuals.append(np.stack(visuals))   # (T, H, W, 3)
        all_proprios.append(np.stack(proprios)) # (T, 2)
        all_actions.append(np.stack(actions))   # (T, 2)

    return {
        'visuals': np.stack(all_visuals),    # (N, T, H, W, 3)
        'proprios': np.stack(all_proprios),  # (N, T, 2)
        'actions': np.stack(all_actions),    # (N, T, 2)
    }


# Collect the dataset
data = collect_trajectories(n_trajs=200, traj_len=20)
print(f"Dataset shapes:")
print(f"  visuals:  {data['visuals'].shape}")
print(f"  proprios: {data['proprios'].shape}")
print(f"  actions:  {data['actions'].shape}")

In [ ]:
# Visualize a few trajectories
fig, axes = plt.subplots(2, 10, figsize=(20, 4))
for row in range(2):
    traj_idx = row
    for col in range(10):
        t = col * 2  # Show every other frame
        axes[row, col].imshow(data['visuals'][traj_idx, t])
        axes[row, col].set_title(f't={t}', fontsize=8)
        axes[row, col].axis('off')
fig.suptitle('Sample Trajectories (every 2nd frame)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. The DINO Encoder (Frozen)

This is the **core insight** of the paper: instead of learning visual representations from scratch, use DINOv2 — a vision transformer pre-trained on millions of images via self-supervised learning.

DINOv2 splits each image into **14×14 pixel patches** and produces a **384-dimensional embedding** for each patch. For a 224×224 image:
- Number of patches = (224/14)² = **256 patches**
- Each patch → 384-dim vector
- Output shape: **(256, 384)**

**Why DINO?** Its features capture semantic structure — it "understands" that the ball is an object, separate from the background. This is far richer than raw pixels and gives the dynamics model a meaningful space to predict in.

**Key: The encoder is FROZEN.** We never update its weights. This is what makes the approach data-efficient.

In [ ]:
class DINOEncoder(nn.Module):
    """Frozen DINOv2 encoder that extracts patch-level features.

    Takes a 224x224 RGB image and returns 256 patch embeddings of dimension 384.
    The encoder weights are FROZEN — we only use DINO as a feature extractor.
    """

    def __init__(self):
        super().__init__()
        # Load DINOv2 ViT-Small with patch size 14
        self.dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        self.dino.eval()

        # Freeze all parameters — this is critical!
        for param in self.dino.parameters():
            param.requires_grad = False

        self.embed_dim = 384   # ViT-Small embedding dimension
        self.patch_size = 14
        self.num_patches = (224 // 14) ** 2  # 256

    @torch.no_grad()
    def forward(self, x):
        """Extract patch features from images.

        Args:
            x: (B, 3, 224, 224) normalized RGB images

        Returns:
            patch_features: (B, 256, 384) patch embeddings
        """
        # DINOv2 forward returns a dict with patch tokens
        features = self.dino.forward_features(x)
        patch_tokens = features['x_norm_patchtokens']  # (B, 256, 384)
        return patch_tokens


# Initialize and test
dino_encoder = DINOEncoder().to(device)
print(f"DINO encoder loaded: {sum(p.numel() for p in dino_encoder.parameters()) / 1e6:.1f}M params (all frozen)")

# Test with a sample image
sample_img = torch.from_numpy(data['visuals'][0, 0]).permute(2, 0, 1).unsqueeze(0).to(device)
with torch.no_grad():
    features = dino_encoder(sample_img)
print(f"Input: {sample_img.shape} → Output: {features.shape}")
print(f"Each image produces {features.shape[1]} patches × {features.shape[2]} dims")

In [ ]:
# Visualize what DINO "sees" — PCA of patch features projected to RGB
from sklearn.decomposition import PCA

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Encode several frames
frames_to_show = [0, 4, 8, 12, 16]
all_features = []
for t in frames_to_show:
    img = torch.from_numpy(data['visuals'][0, t]).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = dino_encoder(img)  # (1, 256, 384)
    all_features.append(feats.cpu().numpy().squeeze())

# Fit PCA on all features together for consistent colors
all_feats_np = np.concatenate(all_features, axis=0)  # (5*256, 384)
pca = PCA(n_components=3)
pca.fit(all_feats_np)

for i, t in enumerate(frames_to_show):
    # Original image
    axes[0, i].imshow(data['visuals'][0, t])
    axes[0, i].set_title(f'Frame t={t}', fontsize=9)
    axes[0, i].axis('off')

    # PCA of DINO features → 16x16 feature map
    feat_pca = pca.transform(all_features[i])  # (256, 3)
    feat_map = feat_pca.reshape(16, 16, 3)
    # Normalize to [0, 1] for display
    feat_map = (feat_map - feat_map.min()) / (feat_map.max() - feat_map.min() + 1e-8)
    axes[1, i].imshow(feat_map)
    axes[1, i].set_title('DINO features', fontsize=9)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Raw Image', fontsize=10)
axes[1, 0].set_ylabel('DINO PCA', fontsize=10)
fig.suptitle('What DINO "Sees": PCA of patch features → RGB', fontsize=12)
plt.tight_layout()
plt.show()

print("Notice how DINO features highlight the ball distinctly from the background.")
print("This semantic understanding is FREE — we didn't train anything!")

---
## 3. Action & Proprioceptive Encoders

The world model needs to know **what action was taken** and **what the robot's state is** (proprioception). These are low-dimensional vectors (2D each in our case), so we project them into small embeddings that get concatenated with the visual patch tokens.

In the paper, these use 1D convolutions, but for our simple case, linear layers work identically.

In [ ]:
class ActionEncoder(nn.Module):
    """Encodes action vectors into embeddings that can be added to the patch sequence.

    The paper uses 1D convolutions; for our 2D action space, a linear layer
    is equivalent. The output becomes additional tokens appended to the
    visual patch sequence.
    """

    def __init__(self, action_dim=2, embed_dim=384, n_tokens=4):
        super().__init__()
        self.n_tokens = n_tokens
        self.embed_dim = embed_dim
        # Project action to n_tokens embeddings of size embed_dim
        self.projection = nn.Sequential(
            nn.Linear(action_dim, 128),
            nn.GELU(),
            nn.Linear(128, n_tokens * embed_dim),
        )

    def forward(self, action):
        """Encode actions into token embeddings.

        Args:
            action: (B, action_dim)
        Returns:
            tokens: (B, n_tokens, embed_dim)
        """
        out = self.projection(action)  # (B, n_tokens * embed_dim)
        return out.reshape(-1, self.n_tokens, self.embed_dim)


class ProprioEncoder(nn.Module):
    """Encodes proprioceptive state (e.g., joint positions) into token embeddings."""

    def __init__(self, proprio_dim=2, embed_dim=384, n_tokens=4):
        super().__init__()
        self.n_tokens = n_tokens
        self.embed_dim = embed_dim
        self.projection = nn.Sequential(
            nn.Linear(proprio_dim, 128),
            nn.GELU(),
            nn.Linear(128, n_tokens * embed_dim),
        )

    def forward(self, proprio):
        """Encode proprioceptive state into token embeddings.

        Args:
            proprio: (B, proprio_dim)
        Returns:
            tokens: (B, n_tokens, embed_dim)
        """
        out = self.projection(proprio)
        return out.reshape(-1, self.n_tokens, self.embed_dim)


# Test the encoders
act_enc = ActionEncoder(action_dim=2, embed_dim=384, n_tokens=4).to(device)
prop_enc = ProprioEncoder(proprio_dim=2, embed_dim=384, n_tokens=4).to(device)

sample_action = torch.randn(1, 2).to(device)
sample_proprio = torch.randn(1, 2).to(device)

act_tokens = act_enc(sample_action)
prop_tokens = prop_enc(sample_proprio)
print(f"Action encoder:  {sample_action.shape} → {act_tokens.shape}")
print(f"Proprio encoder: {sample_proprio.shape} → {prop_tokens.shape}")
print(f"\nThese tokens get concatenated with the {features.shape[1]} visual patch tokens.")
print(f"Total sequence per timestep: {features.shape[1]} + {act_tokens.shape[1]} + {prop_tokens.shape[1]} = {features.shape[1] + act_tokens.shape[1] + prop_tokens.shape[1]} tokens")

---
## 4. The Transformer Predictor (The Dynamics Model)

This is the **heart of the world model** — the component that learns physics.

Given a sequence of past observations (encoded as patch tokens + action/proprio tokens), the predictor uses a **causal Transformer** to predict the embeddings for the **next** timestep.

### Key design choices from the paper:
- **Causal attention mask:** Each timestep can only attend to itself and past timesteps (no peeking at the future)
- **Temporal position embeddings:** Tell the transformer which timestep each token belongs to
- **Patch position embeddings:** Tell it which spatial location each token represents
- Architecture: 6 layers, 16 heads in the paper; we use **4 layers, 8 heads** for speed

The predictor outputs embeddings that should match the DINO encoding of the **actual** next frame.

In [ ]:
class TransformerPredictor(nn.Module):
    """Causal Transformer that predicts next-step embeddings from history.

    Given a sequence of (visual_patches + action_tokens + proprio_tokens) for
    each of `num_hist` timesteps, predicts the visual patch embeddings for
    the next timestep.

    Uses a causal attention mask so each timestep only attends to past.
    """

    def __init__(
        self,
        embed_dim=384,
        depth=4,
        num_heads=8,
        mlp_ratio=4,
        num_patches=256,
        num_extra_tokens=8,  # action_tokens + proprio_tokens per timestep
        num_hist=3,
    ):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_patches = num_patches
        self.num_extra_tokens = num_extra_tokens
        self.tokens_per_step = num_patches + num_extra_tokens
        self.num_hist = num_hist

        # Position embeddings
        # Spatial: which patch position (shared across timesteps)
        self.spatial_pos_embed = nn.Parameter(
            torch.randn(1, self.tokens_per_step, embed_dim) * 0.02
        )
        # Temporal: which timestep (shared across patches)
        self.temporal_pos_embed = nn.Parameter(
            torch.randn(1, num_hist + 1, embed_dim) * 0.02  # +1 for predicted step
        )

        # Transformer layers
        mlp_dim = int(embed_dim * mlp_ratio)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=0.1,
            activation='gelu',
            batch_first=True,
            norm_first=True,  # Pre-norm (more stable training)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

        # Prediction head: project transformer output to predicted patch embeddings
        self.pred_head = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
        )

    def _make_causal_mask(self, seq_len, device):
        """Create a block-causal attention mask.

        Each timestep's tokens can attend to all tokens in the same
        and previous timesteps, but NOT future timesteps.

        Shape: (seq_len, seq_len) with -inf for masked positions.
        """
        # Determine which timestep each token belongs to
        n_steps = seq_len // self.tokens_per_step
        step_ids = torch.arange(seq_len, device=device) // self.tokens_per_step

        # Token i can attend to token j if step_ids[i] >= step_ids[j]
        # IMPORTANT: Use masked_fill_ instead of float() * -inf
        # because 0.0 * float('-inf') = nan in IEEE 754!
        blocked = step_ids.unsqueeze(0) > step_ids.unsqueeze(1)  # True = blocked
        mask = torch.zeros(seq_len, seq_len, device=device)
        mask.masked_fill_(blocked, float('-inf'))
        return mask

    def forward(self, z_sequence):
        """Predict next-step embeddings from a history of embeddings.

        Args:
            z_sequence: (B, num_hist * tokens_per_step, embed_dim)
                Concatenated embeddings for `num_hist` timesteps.
                Each timestep has (num_patches + num_extra_tokens) tokens.

        Returns:
            z_pred: (B, num_patches, embed_dim)
                Predicted visual patch embeddings for the next timestep.
        """
        B, S, D = z_sequence.shape
        n_steps = S // self.tokens_per_step

        # Add spatial position embeddings (repeated for each timestep)
        spatial = self.spatial_pos_embed.repeat(1, n_steps, 1)  # (1, S, D)
        z_sequence = z_sequence + spatial

        # Add temporal position embeddings (same for all tokens in a timestep)
        temp_embeds = self.temporal_pos_embed[:, :n_steps]  # (1, n_steps, D)
        temp_embeds = temp_embeds.repeat_interleave(self.tokens_per_step, dim=1)  # (1, S, D)
        z_sequence = z_sequence + temp_embeds

        # Create causal mask
        mask = self._make_causal_mask(S, z_sequence.device)

        # Run through transformer
        z_out = self.transformer(z_sequence, mask=mask)  # (B, S, D)
        z_out = self.norm(z_out)

        # Extract the last timestep's visual patch tokens
        # (the transformer's prediction for what comes next)
        last_step_start = (n_steps - 1) * self.tokens_per_step
        z_last = z_out[:, last_step_start:last_step_start + self.num_patches]  # (B, 256, D)

        # Apply prediction head
        z_pred = self.pred_head(z_last)  # (B, 256, D)

        return z_pred


# Test the predictor
predictor = TransformerPredictor(
    embed_dim=384, depth=4, num_heads=8, num_patches=256,
    num_extra_tokens=8, num_hist=3
).to(device)

# Simulate 3 timesteps of input, each with 256+8=264 tokens
dummy_seq = torch.randn(2, 3 * 264, 384).to(device)
pred = predictor(dummy_seq)
print(f"Predictor: {sum(p.numel() for p in predictor.parameters()) / 1e6:.1f}M trainable params")
print(f"Input: {dummy_seq.shape} → Predicted: {pred.shape}")

# Verify no NaNs in output
assert not torch.isnan(pred).any(), "NaN detected in predictor output!"
print("✓ No NaN in output — causal mask is correct")

---
## 5. The VQ-VAE Decoder

The predictor outputs embeddings in DINO's feature space. To **visualize** what the model thinks the next frame looks like, we need a **decoder** that maps embeddings back to images.

The paper uses a **Vector-Quantized VAE (VQ-VAE)** decoder:
1. Take the predicted 16×16 grid of 384-dim patch embeddings
2. Quantize them through a learned codebook (discretization)
3. Upsample through transposed convolutions to 224×224

The VQ step helps with training stability and produces sharper images.

**Why not just use a simple decoder?** Vector quantization adds a useful bottleneck — it forces the model to learn a compact, discrete representation, which regularizes training and improves generalization.

In [ ]:
class VectorQuantizer(nn.Module):
    """Vector Quantization layer — the core of VQ-VAE.

    Maintains a codebook of K vectors. Each input embedding is replaced
    by its nearest codebook entry. Uses straight-through estimator for
    gradient flow.

    Losses:
    - Commitment loss: encourages encoder outputs to stay close to codebook
    - Codebook loss: moves codebook entries toward encoder outputs
    """

    def __init__(self, num_codes=512, embed_dim=384, commitment_weight=0.25):
        super().__init__()
        self.num_codes = num_codes
        self.embed_dim = embed_dim
        self.commitment_weight = commitment_weight

        # The codebook: K learnable vectors of dimension D
        self.codebook = nn.Embedding(num_codes, embed_dim)
        # Initialize codebook uniformly
        self.codebook.weight.data.uniform_(-1.0 / num_codes, 1.0 / num_codes)

    def forward(self, z):
        """Quantize continuous embeddings to nearest codebook vectors.

        Args:
            z: (B, H, W, D) continuous embeddings

        Returns:
            z_q: (B, H, W, D) quantized embeddings (straight-through)
            vq_loss: scalar quantization loss
        """
        B, H, W, D = z.shape
        z_flat = z.reshape(-1, D)  # (B*H*W, D)

        # Find nearest codebook entry for each embedding
        # dist(z, e) = ||z||² + ||e||² - 2*z·e
        distances = (
            z_flat.pow(2).sum(dim=1, keepdim=True)
            + self.codebook.weight.pow(2).sum(dim=1)
            - 2 * z_flat @ self.codebook.weight.t()
        )  # (B*H*W, K)

        indices = distances.argmin(dim=1)  # (B*H*W,)
        z_q = self.codebook(indices).reshape(B, H, W, D)  # (B, H, W, D)

        # VQ losses
        codebook_loss = F.mse_loss(z_q.detach(), z)   # Move codebook toward encoder
        commitment_loss = F.mse_loss(z_q, z.detach()) # Move encoder toward codebook
        vq_loss = codebook_loss + self.commitment_weight * commitment_loss

        # Straight-through estimator: copy gradients from z_q to z
        z_q = z + (z_q - z).detach()

        return z_q, vq_loss


class ResBlock(nn.Module):
    """Simple residual block for the decoder."""

    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(8, channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class VQVAEDecoder(nn.Module):
    """Decodes DINO patch embeddings back to images via VQ-VAE.

    Pipeline: patch embeddings (B, 256, 384)
       → reshape to (B, 16, 16, 384)
       → vector quantize
       → upsample via transposed convolutions
       → output image (B, 3, 224, 224)
    """

    def __init__(self, embed_dim=384, num_codes=512):
        super().__init__()
        self.embed_dim = embed_dim

        # Pre-quantization projection
        self.pre_quant = nn.Linear(embed_dim, embed_dim)

        # Vector quantizer
        self.quantizer = VectorQuantizer(num_codes=num_codes, embed_dim=embed_dim)

        # Decoder: upsample from 16x16 to 224x224
        # 16 → 32 → 64 → 128 → 224 (with some careful sizing)
        self.decoder = nn.Sequential(
            # 16x16 → 32x32
            nn.ConvTranspose2d(embed_dim, 256, 4, stride=2, padding=1),
            ResBlock(256),
            # 32x32 → 64x64
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            ResBlock(128),
            # 64x64 → 128x128
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            ResBlock(64),
            # 128x128 → 224x224 (using interpolation for non-power-of-2)
            nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32, 3, 3, padding=1),
            nn.Sigmoid(),  # Output in [0, 1]
        )

    def forward(self, z_patches):
        """Decode patch embeddings to images.

        Args:
            z_patches: (B, 256, 384) patch embeddings

        Returns:
            images: (B, 3, 224, 224) reconstructed images
            vq_loss: scalar VQ quantization loss
        """
        B = z_patches.shape[0]

        # Project and reshape to spatial grid
        z = self.pre_quant(z_patches)         # (B, 256, 384)
        z = z.reshape(B, 16, 16, self.embed_dim)  # (B, 16, 16, 384)

        # Vector quantize
        z_q, vq_loss = self.quantizer(z)     # (B, 16, 16, 384)

        # Rearrange for convolutions: (B, C, H, W)
        z_q = z_q.permute(0, 3, 1, 2)        # (B, 384, 16, 16)

        # Decode to image
        images = self.decoder(z_q)            # (B, 3, 224, 224)

        return images, vq_loss


# Test the decoder
decoder = VQVAEDecoder(embed_dim=384, num_codes=512).to(device)
print(f"Decoder: {sum(p.numel() for p in decoder.parameters()) / 1e6:.1f}M trainable params")

dummy_z = torch.randn(2, 256, 384).to(device)
recon, vq_loss = decoder(dummy_z)
print(f"Input: {dummy_z.shape} → Reconstructed: {recon.shape}, VQ loss: {vq_loss.item():.4f}")

---
## 6. The Complete World Model

Now we assemble all the pieces into a single `DinoWorldModel` class. This is the equivalent of `visual_world_model.py` in the paper's codebase.

### Training forward pass:
1. **Encode** all frames with frozen DINO → patch embeddings
2. **Encode** actions & proprio → small token embeddings
3. **Concatenate** visual + action + proprio tokens for each timestep
4. **Feed** the history sequence to the Transformer predictor → predicted next embeddings
5. **Decode** predicted embeddings → predicted next image
6. **Compute losses**: embedding prediction loss + image reconstruction loss + VQ loss

In [ ]:
class DinoWorldModel(nn.Module):
    """Complete DINO World Model.

    Combines:
    - Frozen DINO encoder (visual feature extraction)
    - Trainable action & proprio encoders
    - Trainable Transformer predictor (dynamics model)
    - Trainable VQ-VAE decoder (image reconstruction)
    """

    def __init__(
        self,
        action_dim=2,
        proprio_dim=2,
        embed_dim=384,
        num_patches=256,
        act_tokens=4,
        prop_tokens=4,
        num_hist=3,
        predictor_depth=4,
        predictor_heads=8,
    ):
        super().__init__()
        self.num_hist = num_hist
        self.num_patches = num_patches
        self.act_tokens = act_tokens
        self.prop_tokens = prop_tokens
        self.tokens_per_step = num_patches + act_tokens + prop_tokens

        # Components
        self.dino_encoder = DINOEncoder()
        self.action_encoder = ActionEncoder(action_dim, embed_dim, act_tokens)
        self.proprio_encoder = ProprioEncoder(proprio_dim, embed_dim, prop_tokens)
        self.predictor = TransformerPredictor(
            embed_dim=embed_dim,
            depth=predictor_depth,
            num_heads=predictor_heads,
            num_patches=num_patches,
            num_extra_tokens=act_tokens + prop_tokens,
            num_hist=num_hist,
        )
        self.decoder = VQVAEDecoder(embed_dim=embed_dim)

    def encode_step(self, visual, action, proprio):
        """Encode a single timestep into a sequence of tokens.

        Args:
            visual: (B, 3, 224, 224)
            action: (B, action_dim)
            proprio: (B, proprio_dim)

        Returns:
            z: (B, tokens_per_step, embed_dim)
            z_visual: (B, num_patches, embed_dim) — just the visual part
        """
        z_visual = self.dino_encoder(visual)       # (B, 256, 384)
        z_action = self.action_encoder(action)     # (B, 4, 384)
        z_proprio = self.proprio_encoder(proprio)  # (B, 4, 384)

        # Concatenate: [visual_patches | action_tokens | proprio_tokens]
        z = torch.cat([z_visual, z_action, z_proprio], dim=1)  # (B, 264, 384)
        return z, z_visual

    def forward(self, visuals, actions, proprios):
        """Training forward pass: predict next frame from history.

        Args:
            visuals: (B, T, 3, 224, 224)  — T = num_hist + 1
            actions: (B, T, action_dim)
            proprios: (B, T, proprio_dim)

        Returns:
            loss_dict with all loss components
        """
        B, T = visuals.shape[:2]
        assert T == self.num_hist + 1, f"Need {self.num_hist + 1} frames, got {T}"

        # Step 1: Encode all timesteps
        all_z = []
        all_z_visual = []
        for t in range(T):
            z_t, z_vis_t = self.encode_step(
                visuals[:, t], actions[:, t], proprios[:, t]
            )
            all_z.append(z_t)
            all_z_visual.append(z_vis_t)

        # Step 2: Build history sequence (first num_hist timesteps)
        z_history = torch.cat(all_z[:self.num_hist], dim=1)  # (B, num_hist*264, 384)

        # Step 3: Predict next visual embeddings
        z_pred = self.predictor(z_history)  # (B, 256, 384)

        # Step 4: Target = DINO encoding of actual next frame
        z_target = all_z_visual[self.num_hist]  # (B, 256, 384)

        # Step 5: Embedding prediction loss (the main dynamics loss)
        z_loss = F.mse_loss(z_pred, z_target.detach())

        # Step 6: Decode predicted embeddings to image
        img_pred, vq_loss_pred = self.decoder(z_pred)

        # Step 7: Image reconstruction loss
        img_target = visuals[:, self.num_hist]  # (B, 3, 224, 224)
        recon_loss = F.mse_loss(img_pred, img_target)

        # Step 8: Also train decoder on ground-truth DINO embeddings
        # (This helps the decoder learn faster — it doesn't depend on predictor quality)
        img_recon_gt, vq_loss_gt = self.decoder(z_target.detach())
        recon_loss_gt = F.mse_loss(img_recon_gt, img_target)

        # Total loss
        total_loss = (
            z_loss                # Embedding prediction (dynamics)
            + recon_loss          # Image reconstruction from predictions
            + 0.25 * vq_loss_pred # VQ-VAE loss on predictions
            + recon_loss_gt       # Image reconstruction from GT embeddings
            + 0.25 * vq_loss_gt   # VQ-VAE loss on GT
        )

        return {
            'total_loss': total_loss,
            'z_loss': z_loss.item(),
            'recon_loss': recon_loss.item(),
            'recon_loss_gt': recon_loss_gt.item(),
            'vq_loss': (vq_loss_pred + vq_loss_gt).item(),
        }

    @torch.no_grad()
    def predict_next(self, visuals, actions, proprios):
        """Predict the next frame given history (for inference).

        Args:
            visuals: (B, num_hist, 3, 224, 224)
            actions: (B, num_hist, action_dim)
            proprios: (B, num_hist, proprio_dim)

        Returns:
            predicted_image: (B, 3, 224, 224)
            z_pred: (B, 256, 384) predicted embeddings
        """
        all_z = []
        for t in range(self.num_hist):
            z_t, _ = self.encode_step(
                visuals[:, t], actions[:, t], proprios[:, t]
            )
            all_z.append(z_t)

        z_history = torch.cat(all_z, dim=1)
        z_pred = self.predictor(z_history)
        img_pred, _ = self.decoder(z_pred)
        return img_pred, z_pred


# Build the complete model
model = DinoWorldModel(
    action_dim=2, proprio_dim=2, num_hist=3,
    predictor_depth=4, predictor_heads=8,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"Total parameters:     {total_params / 1e6:.1f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.1f}M")
print(f"Frozen parameters:    {frozen_params / 1e6:.1f}M (DINO encoder)")
print(f"\nTrainable/Total ratio: {trainable_params/total_params*100:.1f}%")

---
## 7. Dataset & DataLoader

We need to slice our trajectories into training windows. Each training sample is a sequence of `num_hist + 1` consecutive frames (3 history + 1 target).

This is the equivalent of `TrajSlicerDataset` in the paper.

In [ ]:
class TrajectorySliceDataset(torch.utils.data.Dataset):
    """Slices trajectories into overlapping windows for training.

    Each sample is (num_hist + 1) consecutive frames:
    - num_hist frames of history (input)
    - 1 frame to predict (target)
    """

    def __init__(self, data, num_hist=3):
        self.num_hist = num_hist
        self.window = num_hist + 1

        self.visuals = data['visuals']    # (N, T, H, W, 3)
        self.proprios = data['proprios']  # (N, T, 2)
        self.actions = data['actions']    # (N, T, 2)

        N, T = self.visuals.shape[:2]
        # Build index mapping: (traj_idx, start_t)
        self.indices = []
        for i in range(N):
            for t in range(T - self.window + 1):
                self.indices.append((i, t))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        traj_idx, start_t = self.indices[idx]
        end_t = start_t + self.window

        # Get window slices
        vis = self.visuals[traj_idx, start_t:end_t]     # (W, H, W, 3)
        prop = self.proprios[traj_idx, start_t:end_t]   # (W, 2)
        act = self.actions[traj_idx, start_t:end_t]     # (W, 2)

        # Convert images to (W, 3, H, W) float tensor
        vis = torch.from_numpy(vis).permute(0, 3, 1, 2).float()
        prop = torch.from_numpy(prop).float()
        act = torch.from_numpy(act).float()

        return vis, act, prop


# Create dataset and dataloader
dataset = TrajectorySliceDataset(data, num_hist=3)
dataloader = torch.utils.data.DataLoader(
    dataset, batch_size=16, shuffle=True,
    num_workers=0,  # Use 0 on Colab to avoid multiprocessing cleanup errors
    pin_memory=True,
)

print(f"Dataset: {len(dataset)} training windows")
print(f"Batches per epoch: {len(dataloader)}")

# Verify a sample
vis, act, prop = dataset[0]
print(f"\nSample shapes:")
print(f"  visuals:  {vis.shape}  (window=4 frames, CHW format)")
print(f"  actions:  {act.shape}")
print(f"  proprios: {prop.shape}")

---
## 8. Training Loop

Now for the actual training! Following the paper's approach:

- **Separate optimizers** for each trainable component (predictor, decoder, action/proprio encoders)
- **DINO encoder stays frozen** — no gradients flow through it
- **Loss = embedding prediction + image reconstruction + VQ regularization**

We'll train for ~30 epochs, which takes about 10-12 minutes on a T4.

In [ ]:
# Set up optimizers (separate LRs like the paper)
optimizer = torch.optim.AdamW([
    {'params': model.predictor.parameters(), 'lr': 5e-4},
    {'params': model.decoder.parameters(), 'lr': 3e-4},
    {'params': model.action_encoder.parameters(), 'lr': 5e-4},
    {'params': model.proprio_encoder.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

print("Optimizer configured with separate learning rates:")
for i, pg in enumerate(optimizer.param_groups):
    names = ['predictor', 'decoder', 'action_encoder', 'proprio_encoder']
    print(f"  {names[i]:20s}: lr={pg['lr']:.0e}, params={sum(p.numel() for p in pg['params'])/1e6:.1f}M")

In [ ]:
# Training loop
NUM_EPOCHS = 30

history = {'total_loss': [], 'z_loss': [], 'recon_loss': [], 'recon_loss_gt': [], 'vq_loss': []}

model.train()
# Keep DINO in eval mode always (frozen batch norm etc.)
model.dino_encoder.eval()

for epoch in range(NUM_EPOCHS):
    epoch_losses = {k: [] for k in history}

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for vis, act, prop in pbar:
        vis = vis.to(device)    # (B, 4, 3, 224, 224)
        act = act.to(device)    # (B, 4, 2)
        prop = prop.to(device)  # (B, 4, 2)

        # Forward pass
        loss_dict = model(vis, act, prop)

        # Backward pass
        optimizer.zero_grad()
        loss_dict['total_loss'].backward()
        # Gradient clipping (stabilizes training)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        # Record losses
        for k in history:
            val = loss_dict[k].item() if isinstance(loss_dict[k], torch.Tensor) else loss_dict[k]
            epoch_losses[k].append(val)

        pbar.set_postfix({
            'z': f"{loss_dict['z_loss']:.4f}",
            'recon': f"{loss_dict['recon_loss']:.4f}",
        })

    scheduler.step()

    # Record epoch averages
    for k in history:
        history[k].append(np.mean(epoch_losses[k]))

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Total: {history['total_loss'][-1]:.4f} | "
            f"Z-loss: {history['z_loss'][-1]:.4f} | "
            f"Recon: {history['recon_loss'][-1]:.4f} | "
            f"Recon-GT: {history['recon_loss_gt'][-1]:.4f} | "
            f"VQ: {history['vq_loss'][-1]:.4f}"
        )

print("\nTraining complete!")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

plots = [
    ('total_loss', 'Total Loss', '#D97757'),
    ('z_loss', 'Embedding Prediction Loss\n(dynamics learning)', '#5B8CB8'),
    ('recon_loss', 'Image Reconstruction Loss\n(from predictions)', '#7DA488'),
    ('recon_loss_gt', 'GT Reconstruction Loss\n(decoder learning)', '#9B7EC8'),
]

for ax, (key, title, color) in zip(axes, plots):
    ax.plot(history[key], color=color, linewidth=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

fig.suptitle('Training Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Evaluation: Next-Frame Prediction

Let's see how well our world model predicts the next frame! We'll:
1. Give the model 3 history frames + actions
2. Ask it to predict frame 4
3. Compare predicted vs actual

This is the fundamental test — can the model learn "if the ball is here and I push it right, where does it end up?"

In [ ]:
model.eval()

# Pick some test trajectories (last 20 in our dataset)
n_show = 5
fig, axes = plt.subplots(n_show, 5, figsize=(15, 3 * n_show))

for row in range(n_show):
    traj_idx = 180 + row * 4  # Use trajectories from the end
    start_t = 5  # Start from middle of trajectory

    # Get data
    vis = torch.from_numpy(
        data['visuals'][traj_idx, start_t:start_t+4]
    ).permute(0, 3, 1, 2).float().unsqueeze(0).to(device)
    act = torch.from_numpy(
        data['actions'][traj_idx, start_t:start_t+4]
    ).float().unsqueeze(0).to(device)
    prop = torch.from_numpy(
        data['proprios'][traj_idx, start_t:start_t+4]
    ).float().unsqueeze(0).to(device)

    # Predict next frame
    with torch.no_grad():
        pred_img, _ = model.predict_next(vis[:, :3], act[:, :3], prop[:, :3])

    # Display
    for t in range(3):
        axes[row, t].imshow(vis[0, t].permute(1, 2, 0).cpu().numpy())
        axes[row, t].set_title(f'History t={t}', fontsize=9)
        axes[row, t].axis('off')

    axes[row, 3].imshow(pred_img[0].permute(1, 2, 0).cpu().numpy())
    axes[row, 3].set_title('Predicted Next', fontsize=9, color='#D97757', fontweight='bold')
    axes[row, 3].axis('off')

    axes[row, 4].imshow(vis[0, 3].permute(1, 2, 0).cpu().numpy())
    axes[row, 4].set_title('Actual Next', fontsize=9, color='#7DA488', fontweight='bold')
    axes[row, 4].axis('off')

fig.suptitle('Next-Frame Prediction: 3 History Frames → Predicted vs Actual', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Multi-Step Rollout (Autoregressive Prediction)

The real power of a world model is predicting **multiple steps** into the future. We feed the model's own predictions back as input to predict further ahead.

This is **autoregressive rollout** — the same technique used for planning. Errors can compound over time, so this is a stringent test of model quality.

### How rollout works:
```
Step 0: history = [frame_0, frame_1, frame_2] → predict frame_3
Step 1: history = [frame_1, frame_2, pred_3]  → predict frame_4
Step 2: history = [frame_2, pred_3,  pred_4]  → predict frame_5
...
```

In [ ]:
@torch.no_grad()
def autoregressive_rollout(model, init_visuals, init_actions, init_proprios, future_actions, n_steps=8):
    """Roll out the world model autoregressively for multiple steps.

    Args:
        model: DinoWorldModel
        init_visuals: (1, num_hist, 3, 224, 224) initial frames
        init_actions: (1, num_hist, 2) initial actions
        init_proprios: (1, num_hist, 2) initial proprio states
        future_actions: (1, n_steps, 2) actions to take in the future
        n_steps: how many steps to predict ahead

    Returns:
        predicted_frames: list of (1, 3, 224, 224) predicted images
    """
    model.eval()
    num_hist = model.num_hist

    # Start with initial history
    vis_buffer = list(init_visuals[0])      # List of (3, 224, 224)
    act_buffer = list(init_actions[0])      # List of (2,)
    prop_buffer = list(init_proprios[0])    # List of (2,)

    predicted_frames = []

    for step in range(n_steps):
        # Take the last num_hist frames as input
        vis_in = torch.stack(vis_buffer[-num_hist:]).unsqueeze(0).to(device)
        act_in = torch.stack(act_buffer[-num_hist:]).unsqueeze(0).to(device)
        prop_in = torch.stack(prop_buffer[-num_hist:]).unsqueeze(0).to(device)

        # Predict next frame
        pred_img, z_pred = model.predict_next(vis_in, act_in, prop_in)

        predicted_frames.append(pred_img[0].cpu())

        # Add prediction to buffer for next step
        vis_buffer.append(pred_img[0].cpu())
        act_buffer.append(future_actions[0, step].cpu())
        # Use a dummy proprio (in real use, you'd predict this too)
        prop_buffer.append(prop_buffer[-1])  # repeat last known

    return predicted_frames


# Run a multi-step rollout
traj_idx = 150
start_t = 2
n_future = 10

# Initial history
init_vis = torch.from_numpy(
    data['visuals'][traj_idx, start_t:start_t+3]
).permute(0, 3, 1, 2).float().unsqueeze(0)
init_act = torch.from_numpy(
    data['actions'][traj_idx, start_t:start_t+3]
).float().unsqueeze(0)
init_prop = torch.from_numpy(
    data['proprios'][traj_idx, start_t:start_t+3]
).float().unsqueeze(0)

# Future actions (from the actual trajectory)
future_act = torch.from_numpy(
    data['actions'][traj_idx, start_t+3:start_t+3+n_future]
).float().unsqueeze(0)

# Rollout
pred_frames = autoregressive_rollout(
    model, init_vis, init_act, init_prop, future_act, n_steps=n_future
)

# Visualize predicted vs actual
fig, axes = plt.subplots(3, n_future, figsize=(2 * n_future, 6))

for t in range(n_future):
    # History context (first 3 only)
    if t < 3:
        axes[0, t].imshow(data['visuals'][traj_idx, start_t + t])
        axes[0, t].set_title(f'Hist {t}', fontsize=8)
    else:
        axes[0, t].axis('off')
    axes[0, t].axis('off')

    # Predicted
    axes[1, t].imshow(pred_frames[t].permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, t].set_title(f'Pred +{t+1}', fontsize=8, color='#D97757')
    axes[1, t].axis('off')

    # Actual
    actual_t = start_t + 3 + t
    if actual_t < data['visuals'].shape[1]:
        axes[2, t].imshow(data['visuals'][traj_idx, actual_t])
        axes[2, t].set_title(f'Actual +{t+1}', fontsize=8, color='#7DA488')
    axes[2, t].axis('off')

axes[0, 1].set_title('History Context', fontsize=9, fontweight='bold')
axes[1, 0].set_ylabel('Predicted', fontsize=10, color='#D97757')
axes[2, 0].set_ylabel('Actual', fontsize=10, color='#7DA488')
fig.suptitle(f'Autoregressive Rollout: {n_future} Steps into the Future', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: Predictions may get blurrier further into the future — this is expected!")
print("Errors compound with autoregressive rollout.")

---
## 11. Zero-Shot Planning with CEM

This is the **killer feature** of DINO-WM: **zero-shot planning**. Given:
- A **current observation** (where the ball is now)
- A **goal observation** (where we want the ball to be)

The model plans a sequence of actions to reach the goal, **without ever being trained on goal-reaching!** It does this purely by:
1. Imagining future states using the world model
2. Checking if the imagined future matches the goal (in DINO embedding space)
3. Optimizing actions to minimize the embedding distance to the goal

### CEM (Cross-Entropy Method) Planner:
1. Sample many random action sequences
2. Roll each one out through the world model
3. Score by how close the final predicted embedding is to the goal embedding
4. Keep the top-K best sequences
5. Fit a new Gaussian to the top-K
6. Repeat, narrowing down to the best plan

In [ ]:
class CEMPlanner:
    """Cross-Entropy Method planner for zero-shot goal reaching.

    Plans action sequences by:
    1. Sampling random actions from a Gaussian
    2. Rolling each out through the world model
    3. Scoring by embedding distance to goal
    4. Fitting a new Gaussian to the top-K best
    5. Repeating until convergence
    """

    def __init__(
        self,
        model,
        action_dim=2,
        horizon=5,
        num_samples=200,
        topk=20,
        opt_steps=15,
    ):
        self.model = model
        self.action_dim = action_dim
        self.horizon = horizon
        self.num_samples = num_samples
        self.topk = topk
        self.opt_steps = opt_steps

    @torch.no_grad()
    def plan(self, init_visuals, init_actions, init_proprios, goal_visual):
        """Plan an action sequence to reach goal_visual from current state.

        Args:
            init_visuals: (1, num_hist, 3, 224, 224)
            init_actions: (1, num_hist, action_dim)
            init_proprios: (1, num_hist, proprio_dim)
            goal_visual: (1, 3, 224, 224) goal image

        Returns:
            best_actions: (horizon, action_dim) optimized action sequence
            best_cost: scalar, final embedding distance to goal
        """
        self.model.eval()
        H = self.horizon

        # Encode goal image to get target embeddings
        z_goal = self.model.dino_encoder(goal_visual.to(device))  # (1, 256, 384)

        # Initialize action distribution
        mean = torch.zeros(H, self.action_dim, device=device)
        std = torch.ones(H, self.action_dim, device=device) * 1.5

        best_actions = None
        best_cost = float('inf')

        for opt_iter in range(self.opt_steps):
            # Sample action sequences from current distribution
            # (num_samples, horizon, action_dim)
            noise = torch.randn(self.num_samples, H, self.action_dim, device=device)
            actions = mean.unsqueeze(0) + std.unsqueeze(0) * noise
            actions = actions.clamp(-3, 3)  # Clip to reasonable range

            # Evaluate each action sequence
            costs = []
            for i in range(self.num_samples):
                # Simulate rollout for this action sequence
                act_seq = actions[i]  # (horizon, action_dim)
                z_final = self._simulate(init_visuals, init_actions, init_proprios, act_seq)

                # Cost = embedding distance to goal
                cost = F.mse_loss(z_final, z_goal).item()
                costs.append(cost)

            costs = torch.tensor(costs, device=device)

            # Select top-K
            topk_indices = costs.argsort()[:self.topk]
            topk_actions = actions[topk_indices]  # (topk, H, action_dim)

            # Track best
            if costs[topk_indices[0]].item() < best_cost:
                best_cost = costs[topk_indices[0]].item()
                best_actions = actions[topk_indices[0]].clone()

            # Refit distribution to top-K
            mean = topk_actions.mean(dim=0)
            std = topk_actions.std(dim=0).clamp(min=0.1)

        return best_actions.cpu(), best_cost

    @torch.no_grad()
    def _simulate(self, init_visuals, init_actions, init_proprios, action_seq):
        """Simulate a rollout and return final embeddings."""
        num_hist = self.model.num_hist

        # Encode initial history
        all_z = []
        all_z_vis = []
        for t in range(num_hist):
            z_t, z_vis_t = self.model.encode_step(
                init_visuals[:, t].to(device),
                init_actions[:, t].to(device),
                init_proprios[:, t].to(device),
            )
            all_z.append(z_t)
            all_z_vis.append(z_vis_t)

        # Roll out using predicted embeddings
        for step in range(len(action_seq)):
            # Build history from last num_hist entries
            z_history = torch.cat(all_z[-num_hist:], dim=1)
            z_pred = self.model.predictor(z_history)  # (1, 256, 384)

            # Create full token sequence for predicted step
            act_tokens = self.model.action_encoder(action_seq[step:step+1].unsqueeze(0).to(device))
            prop_tokens = self.model.proprio_encoder(
                init_proprios[:, -1].to(device)  # Use last known proprio
            )
            z_full = torch.cat([z_pred, act_tokens, prop_tokens], dim=1)
            all_z.append(z_full)
            all_z_vis.append(z_pred)

        # Return final visual embeddings
        return all_z_vis[-1]  # (1, 256, 384)


print("CEM Planner ready!")

In [ ]:
# Run the planner on a goal-reaching task
planner = CEMPlanner(
    model, action_dim=2, horizon=5,
    num_samples=150, topk=15, opt_steps=12
)

# Set up a goal-reaching scenario
env = BallArena()

# Start position: left side
start_obs = env.reset(pos=[60, 112])
start_img = torch.from_numpy(start_obs['visual']).permute(2, 0, 1).float()

# Goal position: right side
goal_obs = env.reset(pos=[170, 112])
goal_img = torch.from_numpy(goal_obs['visual']).permute(2, 0, 1).float()

# Build initial history (3 frames at start position with zero actions)
init_vis = start_img.unsqueeze(0).repeat(3, 1, 1, 1).unsqueeze(0)  # (1, 3, 3, 224, 224)
init_act = torch.zeros(1, 3, 2)  # No movement in history
init_prop = torch.tensor(start_obs['proprio']).unsqueeze(0).repeat(1, 3, 1)  # (1, 3, 2)

print("Planning action sequence to reach goal...")
planned_actions, cost = planner.plan(
    init_vis, init_act, init_prop,
    goal_img.unsqueeze(0)
)
print(f"Planning complete! Final cost: {cost:.6f}")
print(f"Planned actions: {planned_actions.numpy().round(2)}")

# Execute the plan in the actual environment
env.reset(pos=[60, 112])
execution_frames = [env._get_obs()['visual']]
for a in planned_actions:
    obs = env.step(a.numpy())
    execution_frames.append(obs['visual'])

# Also predict what the world model *thinks* will happen
pred_frames = autoregressive_rollout(
    model, init_vis, init_act, init_prop,
    planned_actions.unsqueeze(0), n_steps=5
)

# Visualize: start → planned trajectory → goal
n_cols = len(execution_frames) + 1  # +1 for goal
fig, axes = plt.subplots(3, n_cols, figsize=(2.5 * n_cols, 7.5))

# Row 0: Actual execution
for t, frame in enumerate(execution_frames):
    axes[0, t].imshow(frame)
    label = 'Start' if t == 0 else f'Step {t}'
    axes[0, t].set_title(label, fontsize=9)
    axes[0, t].axis('off')
axes[0, -1].imshow(goal_obs['visual'])
axes[0, -1].set_title('GOAL', fontsize=9, color='#7DA488', fontweight='bold')
axes[0, -1].axis('off')

# Row 1: World model predictions
axes[1, 0].imshow(start_obs['visual'])
axes[1, 0].set_title('Start', fontsize=9)
axes[1, 0].axis('off')
for t, frame in enumerate(pred_frames):
    axes[1, t+1].imshow(frame.permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, t+1].set_title(f'Pred {t+1}', fontsize=9, color='#D97757')
    axes[1, t+1].axis('off')
axes[1, -1].imshow(goal_obs['visual'])
axes[1, -1].set_title('GOAL', fontsize=9, color='#7DA488', fontweight='bold')
axes[1, -1].axis('off')

# Row 2: Difference visualization
for t in range(n_cols):
    if t < len(execution_frames):
        diff = np.abs(execution_frames[t] - goal_obs['visual']).mean(axis=2)
        axes[2, t].imshow(diff, cmap='hot', vmin=0, vmax=0.5)
        axes[2, t].set_title(f'Diff to goal', fontsize=8)
    axes[2, t].axis('off')

axes[0, 0].set_ylabel('Actual\nExecution', fontsize=10)
axes[1, 0].set_ylabel('World Model\nPrediction', fontsize=10)
axes[2, 0].set_ylabel('Distance\nto Goal', fontsize=10)

fig.suptitle('Zero-Shot Planning with CEM: Reach the Goal!', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. Visualize Embedding Space

One last thing — let's visualize **why DINO features make this work**. We'll encode many ball positions and see how they're organized in DINO's embedding space.

The key insight: DINO features cluster by **semantic meaning** (ball position), creating a smooth space where "close in embedding space" = "close in physical space". This is what makes planning in embedding space work!

In [ ]:
from sklearn.manifold import TSNE

# Generate images with ball at different positions
env = BallArena()
positions = []
embeddings = []

grid_size = 10
for x in np.linspace(30, 194, grid_size):
    for y in np.linspace(30, 194, grid_size):
        obs = env.reset(pos=[x, y])
        img = torch.from_numpy(obs['visual']).permute(2, 0, 1).unsqueeze(0).to(device)
        with torch.no_grad():
            z = model.dino_encoder(img)  # (1, 256, 384)
            # Average pool patch embeddings to get single vector
            z_avg = z.mean(dim=1)  # (1, 384)
        embeddings.append(z_avg.cpu().numpy().squeeze())
        positions.append([x, y])

embeddings = np.array(embeddings)  # (100, 384)
positions = np.array(positions)    # (100, 2)

# t-SNE reduction to 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=15)
emb_2d = tsne.fit_transform(embeddings)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: actual positions colored by x-coordinate
sc1 = axes[0].scatter(
    positions[:, 0], positions[:, 1],
    c=positions[:, 0], cmap='RdYlGn', s=80, edgecolors='white', linewidths=0.5
)
axes[0].set_title('Actual Ball Positions\n(colored by x-coordinate)', fontsize=11)
axes[0].set_xlabel('X position')
axes[0].set_ylabel('Y position')
axes[0].set_aspect('equal')
plt.colorbar(sc1, ax=axes[0], label='X position')

# Right: t-SNE of DINO embeddings, same coloring
sc2 = axes[1].scatter(
    emb_2d[:, 0], emb_2d[:, 1],
    c=positions[:, 0], cmap='RdYlGn', s=80, edgecolors='white', linewidths=0.5
)
axes[1].set_title('DINO Embedding Space (t-SNE)\n(same coloring — notice the structure!)', fontsize=11)
axes[1].set_xlabel('t-SNE dim 1')
axes[1].set_ylabel('t-SNE dim 2')
plt.colorbar(sc2, ax=axes[1], label='X position')

fig.suptitle(
    'DINO preserves spatial structure in embedding space\n'
    'This is why planning in DINO-space works!',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("Key insight: Points that are close in physical space are also close in DINO space.")
print("This smooth, structured embedding space is what enables zero-shot planning!")

---
## Summary: What We Built

### Architecture Recap

| Component | Trainable? | Purpose |
|-----------|------------|--------|
| **DINO Encoder** | Frozen | Extract rich visual features from raw pixels |
| **Action Encoder** | Yes | Project low-dim actions into embedding space |
| **Proprio Encoder** | Yes | Project proprioceptive state into embedding space |
| **Transformer Predictor** | Yes | Learn dynamics: predict next embeddings from history |
| **VQ-VAE Decoder** | Yes | Reconstruct images from embeddings (for visualization) |

### Key Losses

| Loss | What it does |
|------|-------------|
| **Embedding prediction (z_loss)** | Forces predictor to match DINO features of next frame |
| **Image reconstruction** | Forces decoder to produce sharp images |
| **VQ quantization** | Regularizes the decoder's latent space |

### Why This Works

1. **DINO features are semantically meaningful** — nearby points in DINO space correspond to visually similar scenes
2. **Frozen encoder = data efficiency** — we only need to learn dynamics, not perception
3. **Planning in embedding space** — CEM optimizes actions by comparing predicted vs goal embeddings
4. **Zero-shot transfer** — the model never sees goal-reaching during training, yet can plan to arbitrary goals

### Going Further

- **More complex environments:** Try with obstacles, multiple objects, or robotic manipulation
- **Gradient-based planning:** Replace CEM with backpropagation through the world model
- **Longer horizons:** Use frameskipping (action repetition) to plan further ahead
- **Real robot data:** The paper shows this works on real-world robotic tasks!

### References

- [DINO-WM Paper](https://arxiv.org/abs/2411.04983)
- [Code Repository](https://github.com/gaoyuezhou/dino_wm)
- [DINOv2](https://github.com/facebookresearch/dinov2)
- [VQ-VAE Paper](https://arxiv.org/abs/1711.00937)